# Phase 0: data audit

Two checks that need the data, run on Colab:

1. **TUMMHCD writer information.** Does the archive carry writer IDs (folders, file names,
   side files), or a hidden writer grouping (neighbouring files, or same-numbered files in
   different classes, sharing a writer's style)? Writes `results/tummhcd_audit.json`.
2. **Meitei Mayek text corpora.** Downloads the openly licensed native Meitei Mayek sources and
   counts words, distinct words, and how often the ꯢ / ꯏ orthographic rule holds. Writes
   `results/corpus_stats.json`.

Both result files are copied to `WORK/results` on Google Drive. Commit them to the repository
afterwards: every number in the Phase 0 report should come from them.

Needs no GPU. The optional secrets (Colab key icon): `GITHUB_TOKEN` only while the repository
is private, `HF_TOKEN` for FLORES+ (gated: accept its terms on Hugging Face first).

In [ ]:
# Where things are. Change these to match your Drive.
TUMMHCD_ZIP = "/content/drive/MyDrive/tummhcd98/TUMMHCD-TEST-TRAIN.zip"  # the archive as downloaded
TUMMHCD_DIR = None      # or an extracted copy: set this and TUMMHCD_ZIP = None
WORK = "/content/drive/MyDrive/meitei-word-recognition"
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCHES = ["claude/intelligent-euler-rx42yl", "main"]  # the first that exists and has the scripts is used
EXTRA_CORPORA = {}      # corpora you downloaded yourself, name -> file or folder on Drive,
                        # e.g. {"ilci2": "/content/drive/MyDrive/corpora/ILCI-II-Manipuri"}

In [ ]:
import json, os, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")  # only needed while the repository is private
except Exception:
    token = None

def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token or "\0", "***")
    if out.strip():
        print(out)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")

REPO_DIR = "/content/repo"
if not os.path.exists(f"{REPO_DIR}/scripts/audit_tummhcd.py"):
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    for branch in BRANCHES:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"{REPO_DIR}/scripts/audit_tummhcd.py"):
            break
    else:
        raise SystemExit("no branch in BRANCHES has the Phase 0 scripts. git: "
                         + r.stderr.replace(token or "\0", "***"))
    run("git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git")
os.chdir(REPO_DIR)
os.makedirs(f"{WORK}/results", exist_ok=True)
run("git", "log", "-1", "--format=%h %s")

## 1. TUMMHCD writer information

The archive is copied off Drive first (reading 85,000 small files through the Drive mount is slow).
Style features take about half a minute. How to read the output is explained in
`docs/phase0_audit.md`, section "TUMMHCD writer information".

In [ ]:
if TUMMHCD_ZIP:
    local = "/content/TUMMHCD-TEST-TRAIN.zip"
    if not os.path.exists(local):
        shutil.copy(TUMMHCD_ZIP, local)
    source = ["--zip", local]
else:
    source = ["--dir", TUMMHCD_DIR]
run(sys.executable, "scripts/audit_tummhcd.py", *source, "--out", "results/tummhcd_audit.json")
for name in ("tummhcd_audit.json", "tummhcd_audit_duplicates.csv"):
    shutil.copy(f"results/{name}", f"{WORK}/results/")

In [ ]:
audit = json.load(open("results/tummhcd_audit.json"))
print(json.dumps({k: audit[k] for k in ("structure", "sizes", "other_files", "timestamps", "duplicates")},
                 indent=1, ensure_ascii=False)[:6000])
print(json.dumps(audit["names"], indent=1, ensure_ascii=False)[:4000])

## 2. Meitei Mayek text corpora

Each download is tried on its own; a failure is reported and the rest continue.

| Source | Licence |
|---|---|
| Meitei Wikipedia dump (`mniwiki`) | CC BY-SA 4.0 |
| FineWeb-2 `mni_Mtei` and `mni_Mtei_removed` (the part its filters dropped) | ODC-By 1.0 |
| FLORES+ `mni_Mtei` (gated) | CC BY-SA 4.0 |

Only native Meitei Mayek counts: `corpus_stats.py` ignores every other script.

In [ ]:
import glob, urllib.request

CORPORA = "/content/corpora"
os.makedirs(CORPORA, exist_ok=True)
sources = {}

def fetch(name, get):
    try:
        sources[name] = get()
        print(f"{name}: {sources[name]}")
    except Exception as e:
        print(f"{name}: failed ({type(e).__name__}: {e})")

def wikipedia():
    path = f"{CORPORA}/mniwiki-latest-pages-articles.xml.bz2"
    if not os.path.exists(path):
        req = urllib.request.Request(
            "https://dumps.wikimedia.org/mniwiki/latest/mniwiki-latest-pages-articles.xml.bz2",
            headers={"User-Agent": "meitei-word-recognition/0.1 (research; github.com/chingkheinganba231005)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            shutil.copyfileobj(r, f)
    return path

def hub(repo, pattern, folder):
    from huggingface_hub import snapshot_download
    d = snapshot_download(repo, repo_type="dataset", allow_patterns=[pattern], local_dir=f"{CORPORA}/{folder}")
    files = [f for f in glob.glob(f"{d}/**/*", recursive=True) if os.path.isfile(f) and "/.cache/" not in f]
    assert files, f"nothing matched {pattern}"
    return os.path.commonpath(files) if len(files) > 1 else files[0]

fetch("wikipedia", wikipedia)
fetch("fineweb2", lambda: hub("HuggingFaceFW/fineweb-2", "data/mni_Mtei/*", "fineweb2_mni_Mtei"))
fetch("fineweb2_removed", lambda: hub("HuggingFaceFW/fineweb-2", "data/mni_Mtei_removed/*", "fineweb2_mni_Mtei_removed"))
fetch("flores_plus", lambda: hub("openlanguagedata/flores_plus", "*mni_Mtei*", "flores_plus"))
sources.update(EXTRA_CORPORA)

In [ ]:
if not sources:
    raise SystemExit("nothing downloaded and nothing in EXTRA_CORPORA")
run(sys.executable, "scripts/corpus_stats.py", *[f"{k}={v}" for k, v in sources.items()],
    "--out", "results/corpus_stats.json", "--words-out", f"{WORK}/wordlists")
shutil.copy("results/corpus_stats.json", f"{WORK}/results/")

In [ ]:
stats = json.load(open("results/corpus_stats.json"))
print(f"{'source':22} {'words':>10} {'distinct':>9} {'in block':>8} {'rule':>6} {'ꯢ per ꯏ':>8}")
for name, r in stats.items():
    if "error" in r:
        print(f"{name:22} {r['error']}")
        continue
    rule = r["rule_running_words"]
    print(f"{name:22} {r['running_words']:>10,} {r['distinct_words']:>9,} "
          f"{r['share_of_nonspace_chars_in_meitei_block'] or 0:>8.1%} {rule['rule_accuracy'] or 0:>6.1%} "
          f"{rule['i_lonsum_per_i_letter'] or 0:>8}")
print("\nꯢ / ꯏ by what comes before them (running words):")
for name, r in stats.items():
    if "error" in r:
        continue
    print(f"  {name}")
    for row in r["i_contexts_running_words"]["by_kind"]:
        print(f"    {row['before']:26} ꯢ {row['i_lonsum']:>8,}   ꯏ {row['i_letter']:>8,}")
print("\nResults are in", f"{WORK}/results", "- commit results/*.json to the repository.")